# STEP 2 — Duplicate Resolution: BANKING77 Source vs. Translation Collapse

`data_cleaning.py` flagged 2,513 rows sharing (text, label) with another row in the
same split. This notebook proves **which of those are genuine BANKING77 source
duplicates** (not a translation-pipeline bug — the English ticket really does repeat
in `datasets/original-dataset/{train,test}.csv`) **vs. which are translation
collapses** (two *different* English tickets that happened to render to identical
target-language text, which IS a translation-side issue worth fixing).

**The proof mechanism:** `id` in every `{lang}/{split}_labeled.csv` is a 0-based row
index directly into the untouched `original-dataset/{split}.csv` — verified below.
So for any duplicate-text group, checking whether `text_en` differs across members
tells us definitively which case we're in, against the original file, not a guess.

- **True duplicate** (`text_en` identical across the group) -> the ticket is
  redundant in BANKING77 itself -> drop the extra id(s), id-aligned across all five
  languages.
- **Collapse** (`text_en` differs) -> two real, distinct tickets -> reword the
  colliding language's text via LLM so they read distinctly again, matching this
  dataset's existing colloquial register.

All mechanics live in `dedup_common.py` / `clean_common.py` — this notebook is the
review + execution surface.

In [ ]:
import importlib
import pandas as pd

import clean_common as cc
import dedup_common as dd
import data_cleaning as dcl
importlib.reload(cc); importlib.reload(dd); importlib.reload(dcl)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

## 0. Proof of the `id` <-> original-dataset correspondence

If this holds, `text_en` differences are decisive evidence, not assumption.

In [ ]:
for split in ['train', 'test']:
    ok = dd.verify_id_alignment(split)
    print(f'{split}: id-aligned with original-dataset/{split}.csv -> {ok}')

# the one known exception (documented, not a bug): id 2245 in english/test_labeled.csv
# has a deliberate hand-edit (LKR -> USD) vs the raw original test.csv row — see
# classify_test_dataset.py's docstring ("hand-edits already made ... are preserved").
# It is not part of any duplicate group, so it doesn't affect anything below.

## 1. Action A — True BANKING77 duplicates

Groups where the English source ticket (`text_en`) is byte-identical across ids —
these are provably duplicated in the *original* dataset, before any translation
touched them.

In [ ]:
for split in ['train', 'test']:
    groups = dd.true_duplicate_groups(split)
    print(f'{split}: {len(groups)} true-duplicate groups '
          f'({sum(len(g) for g in groups)} rows, '
          f'{len(dd.true_dup_drop_ids(split))} redundant ids to drop)')

In [ ]:
# EVIDENCE — show the actual original-dataset rows for every group, proving the
# duplication predates any translation work. Sourced straight from the untouched
# original-dataset/train.csv, so this holds regardless of whether Action A below
# has already run.
orig_train = dd.load_original('train')
for group in dd.true_duplicate_groups('train'):
    print('duplicate ids:', group)
    for i in group:
        row = orig_train[int(i)]
        print(f"  original-dataset/train.csv row {i}: text={row['text']!r} category={row['category']!r}")
    print()

In [ ]:
# full list of every true-duplicate group, both splits — for the record
rows = []
for split in ['train', 'test']:
    en = {r['id']: r for r in cc.load_rows('english', split)}
    for g in dd.true_duplicate_groups(split):
        for i in sorted(g, key=int):
            rows.append({'split': split, 'group': ','.join(sorted(g, key=int)),
                         'id': i, 'text_en': en[i]['text_en'], 'category': en[i]['category']})
true_dup_df = pd.DataFrame(rows)
true_dup_df

### Apply — drop the redundant ids (keep lowest id per group), all five languages

In [ ]:
result = dd.apply_true_dup_removal(dry_run=True)
print('DRY RUN — ids that would be dropped per split:', result)

In [ ]:
# Apply for real
result = dd.apply_true_dup_removal(dry_run=False)
print('APPLIED — ids dropped per split:', result)
for split in ['train', 'test']:
    n = len(cc.load_rows('english', split))
    print(f'  english/{split}_labeled.csv now has {n} rows')

## 2. Action B — Translation collapses

Groups where the target-language text collides but the English originals genuinely
differ — two distinct tickets, not a dataset duplicate. Recomputed AFTER Action A so
already-dropped ids don't pollute this view.

In [ ]:
collapse = {}
for lang in ['sinhala', 'singlish', 'tamil', 'tamilish']:
    for split in ['train', 'test']:
        try:
            groups = dd.collapse_groups(lang, split)
        except FileNotFoundError:
            continue
        if groups:
            collapse[(lang, split)] = groups

pd.DataFrame([{'lang': l, 'split': s, 'groups': len(g),
               'rows': sum(len(x['members']) for x in g)}
              for (l, s), g in collapse.items()])

In [ ]:
# EVIDENCE — one collapse example: same target text, genuinely different English.
# singlish/sinhala collapse groups are IDENTICAL in count (444 vs 344 train, 64 vs 64 test)
# because singlish is mechanically derived from sinhala (singlishify()) -- fixing
# sinhala and regenerating singlish resolves both together, no separate LLM pass needed.
g = collapse[('sinhala', 'train')][0]
pd.DataFrame(g['members'])[['id', 'text_en', 'text', 'category', 'sentiment', 'priority']]

In [ ]:
# Tamilish collapse count (173) DIFFERS from Tamil's (218) -- confirms Tamilish was
# generated independently (its own LLM pass), not derived from Tamil text like
# Singlish is. So Tamil and Tamilish must each be reworded separately below.
print('sinhala train collapse groups:', len(collapse[('sinhala','train')]))
print('singlish train collapse groups:', len(collapse[('singlish','train')]))
print('tamil train collapse groups:', len(collapse[('tamil','train')]))
print('tamilish train collapse groups:', len(collapse[('tamilish','train')]))

### Plan

Only **Sinhala** and **Tamil** (native script) need LLM rewording directly:
- **Singlish** is regenerated deterministically from the reworded Sinhala via
  `singlishify()` (the same function `generate_singlish.py` already uses) — no
  separate LLM call, and it stays consistent with how Singlish is normally maintained.
- **Tamilish** has no such deterministic link to Tamil in this dataset (it was
  generated by an independent translation pass), so it needs its own LLM reword pass
  on its own collapse groups.

Preview a handful of groups first to sanity-check quality before running the full
batch (344 Sinhala-train + 64 Sinhala-test + 218 Tamil-train + 173 Tamilish-train
groups).

In [ ]:
# PREVIEW — reword the first 5 Sinhala-train collapse groups, don't write anything yet
preview_groups = collapse[('sinhala', 'train')][:5]
for g, result in dd.reword_groups('sinhala', preview_groups, max_workers=4):
    print('BEFORE (shared text):', g['text'])
    for m in g['members']:
        print(f"  id={m['id']}  en={m['text_en']!r}")
    print('AFTER:', result)
    print()

### Full batch — Sinhala (train)

In [ ]:
groups = collapse[('sinhala', 'train')]
results = []
done = 0
for g, r in dd.reword_groups('sinhala', groups, max_workers=4):
    results.append(r)
    done += 1
    if done % 25 == 0 or done == len(groups):
        print(f'{done}/{len(groups)}')
changed = dd.apply_reword_results('sinhala', 'train', groups, results)
n_failed = sum(1 for r in results if not isinstance(r, list))
print(f'sinhala/train: {changed} rows rewritten, {n_failed} groups failed (left as-is)')

### Full batch — Sinhala (test)

In [ ]:
groups = collapse[('sinhala', 'test')]
results = []
done = 0
for g, r in dd.reword_groups('sinhala', groups, max_workers=4):
    results.append(r)
    done += 1
    if done % 25 == 0 or done == len(groups):
        print(f'{done}/{len(groups)}')
changed = dd.apply_reword_results('sinhala', 'test', groups, results)
n_failed = sum(1 for r in results if not isinstance(r, list))
print(f'sinhala/test: {changed} rows rewritten, {n_failed} groups failed (left as-is)')

### Full batch — Tamil (train)

In [ ]:
groups = collapse[('tamil', 'train')]
results = []
done = 0
for g, r in dd.reword_groups('tamil', groups, max_workers=4):
    results.append(r)
    done += 1
    if done % 25 == 0 or done == len(groups):
        print(f'{done}/{len(groups)}')
changed = dd.apply_reword_results('tamil', 'train', groups, results)
n_failed = sum(1 for r in results if not isinstance(r, list))
print(f'tamil/train: {changed} rows rewritten, {n_failed} groups failed (left as-is)')

### Full batch — Tamilish (train)

In [ ]:
groups = collapse[('tamilish', 'train')]
results = []
done = 0
for g, r in dd.reword_groups('tamilish', groups, max_workers=4):
    results.append(r)
    done += 1
    if done % 25 == 0 or done == len(groups):
        print(f'{done}/{len(groups)}')
changed = dd.apply_reword_results('tamilish', 'train', groups, results)
n_failed = sum(1 for r in results if not isinstance(r, list))
print(f'tamilish/train: {changed} rows rewritten, {n_failed} groups failed (left as-is)')

### Regenerate Singlish from the reworded Sinhala (deterministic, no LLM)

In [ ]:
import sys
sys.path.insert(0, '../../datasets/translation')
from singlishify import singlishify

for split in ['train', 'test']:
    si_rows = cc.load_rows('sinhala', split)
    si_text = {r['id']: r['text'] for r in si_rows}
    sl_rows = cc.load_rows('singlish', split)
    changed = 0
    for r in sl_rows:
        new_singlish = singlishify(si_text[r['id']])
        if r['text'] != new_singlish:
            r['text'] = new_singlish
            changed += 1
    cc.save_rows('singlish', split, sl_rows)
    print(f'{split}: regenerated singlish text, {changed} rows changed')

## 2.5. Step 3 — Untranslated rows

`data_cleaning.py` also flagged 18 rows where `text == text_en` verbatim (8 Sinhala,
8 Singlish — the same 8 ids, since Singlish derives from Sinhala — and 2 Tamilish).
All 18 were short label-like phrases (e.g. "Cancel Transaction", "Supported
countries") rather than full sentences, apparently skipped by the original
translation pass. Hand-translated directly into the same colloquial,
code-mixed-English-loanword register as the rest of each dataset, then Singlish
regenerated from the newly-translated Sinhala via `singlishify()` (deterministic,
same as Action B above).

In [ ]:
un = dcl.get_untranslated(dcl.load())
print(f'untranslated rows remaining: {len(un)}')
un[['lang', 'split', 'id', 'text_en', 'text']] if len(un) else 'none — all 18 fixed'

## 3. Re-check & clean up

Recompute collapse groups after all reword passes — should be empty (Sinhala/
Singlish/Tamil/Tamilish train, Sinhala/Singlish test). Tamilish-test and Tamil-test
don't exist yet, so they're naturally excluded.

In [ ]:
remaining = {}
for lang in ['sinhala', 'singlish', 'tamil', 'tamilish']:
    for split in ['train', 'test']:
        try:
            groups = dd.collapse_groups(lang, split)
        except FileNotFoundError:
            continue
        if groups:
            remaining[(lang, split)] = len(groups)
print('remaining collapse groups:', remaining or '(none)')

In [ ]:
# also re-check the exact_duplicates.csv issue from data_cleaning.py is resolved
data = dcl.load()
exact = dcl.get_exact_duplicates(data)
conflicts = dcl.get_conflicting_duplicates(data)
leftover_dup_text = sum(1 for (l, s), df in data.items()
                        for t in [df['text'].str.strip()] if len(t) != t.nunique())
print('remaining conflicting-label groups:', len(conflicts))
print('remaining exact-duplicate rows (true BANKING77 dups still legitimately share text+label):',
      len(exact))

If `remaining` is empty and only the *known* leftover exact-duplicate rows are true
BANKING77 duplicates we intentionally kept a copy of (there shouldn't be any left
after Action A dropped the redundant ids), delete the stale report file.

In [ ]:
cc.delete_report('exact_duplicates.csv')